In [1]:
import os
# Fixer le répertoire de travail sur la racine de ton projet
os.chdir('/Users/7maksacod/Documents/ISM/Master_2/Memoire_big_data_ia')

In [12]:
import pandas as pd
import numpy as np
import re
import unicodedata
import os
from math import radians, cos, sin, asin, sqrt

print("🚀 Début du traitement Zone Silver (Version Finale - Harmonisation & Dédoublonnage)...")

# =====================================================================
# 1. CHARGEMENT ET HARMONISATION DES DATASETS BRUTS
# =====================================================================

def safe_read(path):
    if not os.path.exists(path):
        print(f"⚠️ Fichier introuvable : {path}")
        return pd.DataFrame()
    try:
        return pd.read_csv(path, sep=';', on_bad_lines='skip', engine='python')
    except:
        return pd.read_csv(path, sep=',', on_bad_lines='skip', engine='python')

df_coin = safe_read('data/raw/coinafrique_deep_dataset.csv')
df_expat = safe_read('data/raw/expatdakar_bronze_dataset.csv')
df_whatsapp = safe_read('data/raw/whatsapp_bronze_dataset.csv')
df_facebook = safe_read('data/raw/annonces_facebook_marketplace.csv')

# --- Assignation stricte des sources ---
if not df_coin.empty and 'source' not in df_coin.columns: df_coin['source'] = 'CoinAfrique'
if not df_expat.empty and 'source' not in df_expat.columns: df_expat['source'] = 'ExpatDakar'
if not df_whatsapp.empty and 'source' not in df_whatsapp.columns: df_whatsapp['source'] = 'WhatsApp'
if not df_facebook.empty and 'source' not in df_facebook.columns: df_facebook['source'] = 'Facebook Marketplace'

# --- Alignement des colonnes du dataset Facebook ---
if not df_facebook.empty:
    # On nettoie de façon agressive tous les caractères invisibles (BOM) des noms de colonnes
    df_facebook.columns = [col.encode('ascii', 'ignore').decode('ascii').strip() for col in df_facebook.columns]
    
    # Mapping vers les colonnes standards
    rename_fb = {
        'prix_affiche_fcfa': 'prix_fcfa',
        'url_annonce': 'url',
        'adresse_complete': 'adresse'
    }
    df_facebook = df_facebook.rename(columns=rename_fb)
    
    # On force la fusion du texte dans 'description_complete' s'il n'existe pas
    # Utilisation d'un filet de sécurité : on cherche la colonne qui contient 'titre'
    col_titre = [col for col in df_facebook.columns if 'titre' in col.lower()]
    
    if 'description_complete' not in df_facebook.columns:
        if col_titre:
            df_facebook['description_complete'] = df_facebook[col_titre[0]].fillna('')
        else:
            df_facebook['description_complete'] = '' # Sécurité absolue si le titre disparaît

df_brut = pd.concat([df_coin, df_expat, df_whatsapp, df_facebook], ignore_index=True)
print(f"📦 Total d'annonces brutes ingérées : {len(df_brut)}")

# =====================================================================
# 2. DICTIONNAIRES & UTILITAIRES SPATIAUX (BRT Retiré)
# =====================================================================

def enlever_accents(texte):
    if pd.isna(texte): return ""
    texte = str(texte).lower()
    return ''.join(c for c in unicodedata.normalize('NFD', texte) if unicodedata.category(c) != 'Mn')

# RETRAIT DU BRT
POINTS_INTERET = {
    'plateau': (14.668, -17.435),
    'vdn': (14.720, -17.465),
    'mer': (14.745, -17.518)
}

COORDS_QUARTIERS = {
    # 💎 Zone Luxe
    'plateau': ('Plateau', 14.668, -17.435), 
    'fann': ('Fann', 14.688, -17.465), 
    'point e': ('Point E', 14.693, -17.456), 'pointe e': ('Point E', 14.693, -17.456),
    'mermoz': ('Mermoz', 14.708, -17.472),
    
    # 🏡 Les "Sacré-Cœur" et "Keur Gorgui"
    'sacre coeur 3': ('Sacré-Cœur 3', 14.718, -17.462), 
    'sacre coeur 2': ('Sacré-Cœur 2', 14.714, -17.460),
    'sacre coeur 1': ('Sacré-Cœur 1', 14.710, -17.458), 
    'sacre coeur': ('Sacré-Cœur', 14.712, -17.458),
    'keur gorgui': ('Cité Keur Gorgui', 14.715, -17.465), 
    
    # 🏘️ Les "Liberté"
    'liberte 6 extension': ('Liberté 6 Extension', 14.725, -17.450), 
    'liberte 6': ('Liberté 6', 14.720, -17.448), 'liberte6': ('Liberté 6', 14.720, -17.448),
    'liberte 5': ('Liberté 5', 14.716, -17.446), 'liberte5': ('Liberté 5', 14.716, -17.446),
    'liberte 4': ('Liberté 4', 14.713, -17.444), 'liberte4': ('Liberté 4', 14.713, -17.444),
    'liberte 3': ('Liberté 3', 14.710, -17.442), 'liberte3': ('Liberté 3', 14.710, -17.442),
    'liberte 2': ('Liberté 2', 14.708, -17.440), 'liberte2': ('Liberté 2', 14.708, -17.440),
    'liberte 1': ('Liberté 1', 14.706, -17.438), 'liberte1': ('Liberté 1', 14.706, -17.438),
    'liberte': ('Liberté', 14.710, -17.445),
    
    # 🏙️ Zone Centre
    'medina': ('Médina', 14.680, -17.442), 
    'fass': ('Fass', 14.685, -17.450), 'fasse': ('Fass', 14.685, -17.450),
    'gueule tapee': ('Gueule Tapée', 14.688, -17.455), 'geule tape': ('Gueule Tapée', 14.688, -17.455),
    'colobane': ('Colobane', 14.690, -17.440),
    'baobab': ('Baobab', 14.705, -17.460),
    'amitie 3': ('Amitié 3', 14.705, -17.455), 'amitie 2': ('Amitié 2', 14.702, -17.452), 
    'amitie 1': ('Amitié 1', 14.698, -17.450), 'amitie': ('Amitié', 14.702, -17.455),
    
    # 🏢 Axe Grand Yoff / Dieuppeul / Castors
    'dieuppeul': ('Dieuppeul', 14.718, -17.448), 'dieupeul': ('Dieuppeul', 14.718, -17.448),
    'derkle': ('Derklé', 14.722, -17.443), 'derkler': ('Derklé', 14.722, -17.443),
    'castors': ('Castors', 14.712, -17.440), 'castor': ('Castors', 14.712, -17.440),
    'grand dakar': ('Grand Dakar', 14.700, -17.445), 
    'niary tally': ('Niary Tally', 14.698, -17.440), 'ben tally': ('Niary Tally', 14.698, -17.440),
    'scat urbam': ('Scat Urbam', 14.732, -17.448),
    'zone de captage': ('Zone de Captage', 14.725, -17.438), 'captage': ('Zone de Captage', 14.725, -17.438),
    'khar yallah': ('Khar Yallah', 14.728, -17.448),
    'grand yoff': ('Grand Yoff', 14.730, -17.445), 'hlm grand yoff': ('HLM Grand Yoff', 14.735, -17.442), 
    'hlm': ('HLM', 14.708, -17.435),

    # ✈️ Foires, VDN et Yoff
    'sicap foire': ('Sicap Foire', 14.735, -17.460), 'nord foire': ('Nord Foire', 14.745, -17.455), 
    'ouest foire': ('Ouest Foire', 14.742, -17.475), 'sud foire': ('Sud Foire', 14.738, -17.465), 
    'yoff': ('Yoff', 14.758, -17.463), 'vdn': ('VDN', 14.720, -17.465),

    # 🏖️ Péninsule
    'ouakam comico': ('Ouakam Comico', 14.725, -17.490), 'ouakam': ('Ouakam', 14.723, -17.487), 'oukam': ('Ouakam', 14.723, -17.487),
    'mamelles': ('Mamelles', 14.730, -17.500), 'mamelle': ('Mamelles', 14.730, -17.500),
    'almadies': ('Almadies', 14.745, -17.518), 'almadie': ('Almadies', 14.745, -17.518), 
    'ngor': ('Ngor', 14.752, -17.515), 'virage': ('Virage', 14.752, -17.495),

    # 🏘️ Cités Spécifiques & Hann
    'cite damel': ('Cité Damel', 14.745, -17.458),
    'sipres': ('Sipres', 14.732, -17.465),
    'cite biagui': ('Cité Biagui', 14.750, -17.470),
    'hann bel air': ('Hann Bel-Air', 14.715, -17.430),

    # 🛣️ Périphérie & Banlieue
    'maristes': ('Maristes', 14.725, -17.432), 'mariste': ('Maristes', 14.725, -17.432),
    'patte d\'oie': ('Patte d\'Oie', 14.735, -17.440), 'patte d oie': ('Patte d\'Oie', 14.735, -17.440),
    'parcelles assainies': ('Parcelles Assainies', 14.760, -17.430), 'parcelles': ('Parcelles', 14.760, -17.430),
    'camberene': ('Cambérène', 14.765, -17.420), 'guediawaye': ('Guédiawaye', 14.770, -17.390), 
    'pikine': ('Pikine', 14.755, -17.390), 'rufisque': ('Rufisque', 14.716, -17.273),
    'keur massar': ('Keur Massar', 14.770, -17.310), 'zac mbao': ('Zac Mbao', 14.745, -17.340), 
    'mbao': ('Mbao', 14.745, -17.340)
}

QUARTIERS_TRIES = sorted(COORDS_QUARTIERS.keys(), key=len, reverse=True)

def distance_haversine(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1): return np.nan
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return round(2 * R * asin(sqrt(a)), 2)

# =====================================================================
# 3. MOTEUR NLP D'EXTRACTION (Silver Processing)
# =====================================================================

def traiter_ligne_corrige(row):
    text_clean = str(row.get('description_complete', '')).lower()
    texte_brut = str(row.get('description_complete', '')) + " " + str(row.get('titre', ''))
    texte_brut_lower = texte_brut.lower()
    
    texte_brut_norm = enlever_accents(texte_brut)
    texte_brut_norm = texte_brut_norm.replace("-", " ").replace("'", " ")
    texte_brut_norm = re.sub(r'\b0(\d)\b', r'\1', texte_brut_norm) 

    # --- A. Prix ---
    prix = row.get('prix_fcfa')
    if pd.isna(prix) or str(prix).strip() == '' or float(prix) > 6000000:
        p_match = re.search(r'([\d\s\u202f]+)\s*(?:F\s*Cfa|CFA|fcfa)', texte_brut, re.IGNORECASE)
        if p_match:
            raw_p = p_match.group(1).strip()
            lines = [l for l in re.split(r'\r|\n', raw_p) if l.strip()]
            if lines:
                p_str = re.sub(r'[^\d]', '', lines[-1])
                prix = int(p_str) if p_str else np.nan
        else:
            w_match = re.search(r'(\d{2,4})(?:mille|mil|k|000)', texte_brut_lower)
            if w_match:
                prix = int(w_match.group(1)) * 1000
            else:
                w_match_exact = re.search(r'(\d{5,7})', texte_brut_lower.replace(' ', '').replace('.', ''))
                if w_match_exact:
                    prix = int(w_match_exact.group(1))

    # --- B. Pièces ---
    pieces = row.get('nb_pieces')
    if pd.isna(pieces):
        p_m = re.search(r'(\d+)\s*(?:chambre|chb|pièce)s?|chambres?\s*:\s*(\d+)', texte_brut, re.IGNORECASE)
        if p_m: 
            pieces = int(p_m.group(1) or p_m.group(2))
        else:
            f_m = re.search(r'\bF(\d)\b', texte_brut, re.IGNORECASE)
            if f_m: pieces = int(f_m.group(1))
            elif any(w in texte_brut_lower for w in ['studio', 'chambre salon']): pieces = 1 

    # --- C. Bains ---
    bains = row.get('nb_salles_bain')
    if pd.isna(bains):
        b_m = re.search(r'(\d+)\s*(?:salle[s]?\s*de\s*bain|douche|toilette)s?|salles?\s*de\s*bain\s*:\s*(\d+)', texte_brut, re.IGNORECASE)
        if b_m: bains = int(b_m.group(1) or b_m.group(2))
        elif 'douche' in texte_brut_lower or 'toilette' in texte_brut_lower: bains = 1

    ratio_bains_chambres = round(bains / pieces, 2) if (pd.notna(bains) and pd.notna(pieces) and pieces > 0) else np.nan

    # --- D. Equipements ---
    has_groupe = int(any(w in text_clean for w in ['groupe', 'électrogène', 'electrogene', 'solaire']))
    has_reserve_eau = int(any(w in text_clean for w in ['réserve d\'eau', 'reserve d\'eau', 'citerne', 'surpresseur', 'suppresseur']))
    is_studio = int(any(w in texte_brut_lower for w in ['studio', 'f1', 'chambre salon']))
    is_villa = int(any(w in texte_brut_lower for w in ['villa', 'duplex', 'triplex', 'maison']))
    has_piscine = int(any(w in text_clean for w in ['piscine', 'pool']))
    has_ascenseur = int(any(w in text_clean for w in ['ascenseur', 'elevator']))
    has_parking = int(any(w in text_clean for w in ['parking', 'garage', 'sous-sol', 'sous sol']))
    has_securite = int(any(w in text_clean for w in ['gardiennage', 'sécurité', 'securite', 'vidéosurveillance', 'h24', '24/24', 'vigile', 'gardien']))
    has_clim = int(any(w in text_clean for w in ['climatiseur', 'climatisation', 'clim', 'split']))
    has_vue_mer = int(any(w in text_clean for w in ['vue mer', 'vue sur mer', 'bord de mer', 'facing mer']))
    is_meuble = int(any(w in text_clean for w in ['meublé', 'meuble', 'équipé', 'equipe']))
    has_cuisine_equipee = int(any(w in text_clean for w in ['cuisine équipée', 'cuisine equipee', 'four', 'hotte']))
    has_balcon = int(any(w in text_clean for w in ['balcon', 'terrasse', 'patio', 'espace familial']))
    is_agence_pro = int(any(w in text_clean for w in ['agence', 'cabinet', 'immobilier', 'sarl', 'suarl', 'courtier']))
    longeur_desc = len(text_clean.split())

    # --- E. Quartier & Distances (SANS LE BRT) ---
    quartier_trouve = 'Inconnu'
    lat, lon = row.get('latitude', np.nan), row.get('longitude', np.nan)
    
    # NLP si la latitude n'existait pas déjà (cas de FB marketplace)
    if pd.isna(lat) or pd.isna(lon):
        for q_nom in QUARTIERS_TRIES:
            if q_nom in texte_brut_norm:
                quartier_trouve, lat, lon = COORDS_QUARTIERS[q_nom]
                break
    else:
        # Si Facebook a fourni la coordonnée, on attribue le quartier
        quartier_trouve = row.get('quartier_clean', 'Inconnu')

    dist_plateau = distance_haversine(lat, lon, POINTS_INTERET['plateau'][0], POINTS_INTERET['plateau'][1])
    dist_vdn = distance_haversine(lat, lon, POINTS_INTERET['vdn'][0], POINTS_INTERET['vdn'][1])
    dist_mer = distance_haversine(lat, lon, POINTS_INTERET['mer'][0], POINTS_INTERET['mer'][1])

    return pd.Series([
        prix, pieces, bains, ratio_bains_chambres, quartier_trouve, lat, lon,
        dist_plateau, dist_vdn, dist_mer,
        has_groupe, has_reserve_eau, is_studio, is_villa,
        has_piscine, has_ascenseur, has_parking, has_securite,
        has_clim, has_vue_mer, is_meuble, has_cuisine_equipee, has_balcon,
        is_agence_pro, longeur_desc
    ])

cols_silver = [
    'prix_fcfa', 'nb_pieces', 'nb_salles_bain', 'ratio_bains_chambres', 'quartier', 'latitude', 'longitude',
    'dist_plateau_km', 'dist_vdn_km', 'dist_mer_km', 
    'has_groupe', 'has_reserve_eau', 'is_studio', 'is_villa',
    'has_piscine', 'has_ascenseur', 'has_parking', 'has_securite',
    'has_clim', 'has_vue_mer', 'is_meuble', 'has_cuisine_equipee', 'has_balcon',
    'is_agence_pro', 'longueur_description'
]

df_brut[cols_silver] = df_brut.apply(traiter_ligne_corrige, axis=1)

# Filtre de marché raisonnable
df_silver = df_brut.dropna(subset=['prix_fcfa']).copy()
df_silver['prix_fcfa'] = pd.to_numeric(df_silver['prix_fcfa'], errors='coerce')
df_silver = df_silver[(df_silver['prix_fcfa'] >= 30000) & (df_silver['prix_fcfa'] <= 3500000)]

# =====================================================================
# 4. SAUVEGARDES ZONE SILVER
# =====================================================================

os.makedirs("data/processed", exist_ok=True)

# A. Sauvegarde du dataset principal propre
fichier_sortie_principal = "data/processed/dataset_dakar_immo.csv"
df_silver.to_csv(fichier_sortie_principal, index=False, sep=";")

# B. Extraction et sauvegarde des annonces "Inconnu"
df_inconnus = df_silver[df_silver['quartier'] == 'Inconnu'].copy()
colonnes_lecture = ['source', 'prix_fcfa', 'titre', 'description_complete']
# Remplir les NA de titre si on l'a importé depuis facebook
if 'titre' in df_inconnus.columns:
    df_inconnus['titre'] = df_inconnus['titre'].fillna('')
else:
    df_inconnus['titre'] = ''

fichier_sortie_inconnus = "data/processed/annonces_inconnues_a_nettoyer.csv"
df_inconnus[colonnes_lecture].to_csv(fichier_sortie_inconnus, index=False, sep=";")

print("\n📊 RÉPARTITION FINALE DES QUARTIERS APRÈS CORRECTION GÉOSPATIALE :")
print(df_silver['quartier'].value_counts().head(15))
print(f"\n✅ REUSSITE : {len(df_silver)} annonces sauvegardées dans '{fichier_sortie_principal}'.")
print(f"🔍 INSPECTION : {len(df_inconnus)} annonces à corriger isolées dans '{fichier_sortie_inconnus}'.")

🚀 Début du traitement Zone Silver (Version Finale - Harmonisation & Dédoublonnage)...
📦 Total d'annonces brutes ingérées : 3992

📊 RÉPARTITION FINALE DES QUARTIERS APRÈS CORRECTION GÉOSPATIALE :
quartier
Almadies       462
Sacré-Cœur     363
Mamelles       317
Point E        278
Inconnu        227
Plateau        172
Ouakam         152
Mermoz         151
Ouest Foire    105
Maristes       101
Ngor            99
Yoff            96
Liberté 6       93
Cité Biagui     75
Dieuppeul       71
Name: count, dtype: int64

✅ REUSSITE : 3728 annonces sauvegardées dans 'data/processed/dataset_dakar_immo.csv'.
🔍 INSPECTION : 227 annonces à corriger isolées dans 'data/processed/annonces_inconnues_a_nettoyer.csv'.


In [13]:
import pandas as pd
import numpy as np
import os
import re

print("🛟 Début du script de rattrapage des annonces inconnues...")

# =====================================================================
# 1. CHARGEMENT DES FICHIERS
# =====================================================================

PATH_MAIN = "data/processed/dataset_dakar_immo.csv"
PATH_UNKNOWN = "data/processed/annonces_inconnues_a_nettoyer.csv"

def safe_read_csv(path):
    try:
        return pd.read_csv(path, sep=';', on_bad_lines='skip', engine='python')
    except Exception:
        return pd.read_csv(path, sep=',', on_bad_lines='skip', engine='python')

df_main = safe_read_csv(PATH_MAIN) if os.path.exists(PATH_MAIN) else pd.DataFrame()
df_unknown = safe_read_csv(PATH_UNKNOWN)

# Normalisation des colonnes
if not df_main.empty:
    df_main.columns = df_main.columns.str.strip().str.lower()
if not df_unknown.empty:
    df_unknown.columns = df_unknown.columns.str.strip().str.lower()

# =====================================================================
# 2. FONCTION D'EXTRACTION GÉOSPATIALE AVANCÉE (Correction Regex)
# =====================================================================

def resolve_quartier(row):
    titre = str(row.get('titre', '')) if pd.notnull(row.get('titre')) else ""
    desc = str(row.get('description_complete', '')) if pd.notnull(row.get('description_complete')) else ""
    text = f"{titre} {desc}".lower()

    # --- A. QUARTIERS ET CITÉS DAKAR & BANLIEUE ---
    if re.search(r'sacré[- ]cœur|sacré[- ]coeur|sacre[- ]coeur|boulangerie jaune|pape diop|good rade', text):
        return 'Sacré-Cœur'
    if re.search(r'parcelle', text):
        return 'Parcelles Assainies'
    if re.search(r'cité keur gor?gui', text):
        return 'Cité Keur Gorgui'
    if re.search(r'sica?f foire|sicap foire', text):
        return 'Sicap Foire'
    if re.search(r'niarr?y[- ]tally|marché samedi', text):
        return 'Niarry Tally'
    if re.search(r'bè?n[ee][- ]tally|bene tally', text):
        return 'Bène Tally'
    if re.search(r'cité mixt?a', text):
        return 'Cité Mixta'
    if re.search(r'hann? marina', text):
        return 'Hann Marinas'
    if re.search(r'kounoune', text):
        return 'Kounoune'
    if re.search(r'dalifort', text):
        return 'Dalifort'
    if re.search(r'tivaou?ane peulh|tivaoune peulh|niakoul rab', text):
        return 'Tivaouane Peulh'
    if re.search(r'\bmalika\b', text):
        return 'Malika'
    if re.search(r'djily mbaye', text):
        return 'Cité Djily Mbaye'
    if re.search(r'émergence|adoha', text):
        return 'Cité de l\'Émergence'
    if re.search(r'sangalkam|cité kalia', text):
        return 'Sangalkam'
    if re.search(r'ndiakhirate', text):
        return 'Ndiakhirate'
    if re.search(r'amadou assane ndoye|centre-ville', text):
        return 'Plateau'
    if re.search(r'cité isra', text):
        return 'Cité ISRA'
    if re.search(r'cité magistrat', text):
        return 'Cité Magistrat'
    if re.search(r'cité des assureurs', text):
        return 'Cité des Assureurs'
    if re.search(r'\bapix\b', text):
        return 'APIX'
    if re.search(r'mbackiyoufaye', text):
        return 'Cité Mbackiyoufaye'
    if re.search(r'rond-point 6', text):
        return 'Liberté 6'
    if re.search(r'cité ataya', text):
        return 'Cité Ataya'
    if re.search(r'grand dakar', text):
        return 'Grand Dakar'

    # --- B. VILLES HORS RÉGION DAKAR (POUR EXCLUSION) ---
    if re.search(r'\b(thiès|thies)\b', text):
        return 'Hors Dakar (Thiès)'
    if re.search(r'\bngaparou\b', text):
        return 'Hors Dakar (Ngaparou)'
    if re.search(r'\b(mbour|m’bour|malicounda)\b', text):
        return 'Hors Dakar (Mbour)'
    if re.search(r'\bsaly\b', text):
        return 'Hors Dakar (Saly)'
    if re.search(r'\bnguerigne\b', text):
        return 'Hors Dakar (Nguerigne)'
    if re.search(r'\bsomone\b', text):
        return 'Hors Dakar (Somone)'
    if re.search(r'\bkaolack\b', text):
        return 'Hors Dakar (Kaolack)'
    if re.search(r'\bsaint-louis\b', text):
        return 'Hors Dakar (Saint-Louis)'

    return 'Inconnu'

# =====================================================================
# 3. APPLICATION ET RE-SAUVEGARDE
# =====================================================================

# Application de la fonction
df_unknown['quartier'] = df_unknown.apply(resolve_quartier, axis=1)

# Séparation des jeux selon le résultat du rattrapage
df_recovered = df_unknown[~df_unknown['quartier'].str.startswith('Hors Dakar') & (df_unknown['quartier'] != 'Inconnu')].copy()
df_hors_dakar = df_unknown[df_unknown['quartier'].str.startswith('Hors Dakar')].copy()
df_still_unknown = df_unknown[df_unknown['quartier'] == 'Inconnu'].copy()

# Concaténation des annonces rattrapées au dataset principal
df_final = pd.concat([df_main, df_recovered], ignore_index=True)

# Sauvegarde
os.makedirs("data/processed", exist_ok=True)
df_final.to_csv("data/processed/dataset_dakar_immo.csv", sep=';', index=False)
df_still_unknown.to_csv("data/processed/annonces_inconnues_restantes.csv", sep=';', index=False)

# Bilan
print(f"✅ {len(df_recovered)} annonces rattrapées et ajoutées au dataset principal.")
print(f"🚫 {len(df_hors_dakar)} annonces Hors Dakar écartées.")
print(f"⚠️ {len(df_still_unknown)} annonces non identifiées isolées dans 'annonces_inconnues_restantes.csv'.")
print(f"\n📦 Nouveau total du dataset dakarois (Silver complet) : {len(df_final)} annonces.")

🛟 Début du script de rattrapage des annonces inconnues...
✅ 110 annonces rattrapées et ajoutées au dataset principal.
🚫 35 annonces Hors Dakar écartées.
⚠️ 82 annonces non identifiées isolées dans 'annonces_inconnues_restantes.csv'.

📦 Nouveau total du dataset dakarois (Silver complet) : 3838 annonces.


In [14]:
import pandas as pd
import numpy as np
import re
import unicodedata
import os
from math import radians, cos, sin, asin, sqrt

print("🧹 Début du nettoyage final (Zone Gold)...")

# =====================================================================
# 1. CHARGEMENT ROBUSTE DU DATASET SILVER
# =====================================================================

PATH_SILVER = 'data/processed/dataset_dakar_immo.csv'
PATH_FALLBACK = 'dataset_dakar_immo.csv'

if os.path.exists(PATH_SILVER):
    fichier_entree = PATH_SILVER
elif os.path.exists(PATH_FALLBACK):
    fichier_entree = PATH_FALLBACK
else:
    raise FileNotFoundError("⚠️ Impossible de trouver 'dataset_dakar_immo.csv'. Exécutez d'abord le script Silver.")

df = pd.read_csv(fichier_entree, sep=';', on_bad_lines='skip', engine='python')
print(f"📦 Annonces ingérées depuis la Zone Silver ({fichier_entree}) : {len(df)}")

# =====================================================================
# 2. NETTOYAGE DES MÉTADONNÉES INUTILES POUR LE ML
# =====================================================================

df = df.drop(columns=['adresse', 'surface_m2'], errors='ignore')

# =====================================================================
# 3. UTILITAIRES SPATIAUX ET DICTIONNAIRE (BRT RETIRÉ)
# =====================================================================

def enlever_accents(texte):
    if pd.isna(texte): return ""
    texte = str(texte).lower()
    return ''.join(c for c in unicodedata.normalize('NFD', texte)
                   if unicodedata.category(c) != 'Mn')

# POINTS D'INTÉRÊT STABLES (Sans BRT)
POINTS_INTERET = {
    'plateau': (14.668, -17.435),
    'vdn': (14.720, -17.465),
    'mer': (14.745, -17.518)
}

COORDS_QUARTIERS_EXTENDED = {
    'plateau': ('Plateau', 14.668, -17.435), 'fann': ('Fann', 14.688, -17.465), 
    'point e': ('Point E', 14.693, -17.456), 'mermoz': ('Mermoz', 14.708, -17.472),
    'sacre coeur 3': ('Sacré-Cœur 3', 14.718, -17.462), 'sacre coeur 2': ('Sacré-Cœur 2', 14.714, -17.460),
    'sacre coeur 1': ('Sacré-Cœur 1', 14.710, -17.458), 'sacre coeur': ('Sacré-Cœur', 14.712, -17.458),
    'keur gorgui': ('Cité Keur Gorgui', 14.715, -17.465), 'liberte 6 extension': ('Liberté 6 Extension', 14.725, -17.450), 
    'liberte 6': ('Liberté 6', 14.720, -17.448), 'liberte 5': ('Liberté 5', 14.716, -17.446),
    'liberte 4': ('Liberté 4', 14.713, -17.444), 'liberte 3': ('Liberté 3', 14.710, -17.442),
    'liberte 2': ('Liberté 2', 14.708, -17.440), 'liberte 1': ('Liberté 1', 14.706, -17.438),
    'liberte': ('Liberté', 14.710, -17.445), 'medina': ('Médina', 14.680, -17.442), 
    'fass': ('Fass', 14.685, -17.450), 'gueule tapee': ('Gueule Tapée', 14.688, -17.455),
    'colobane': ('Colobane', 14.690, -17.440), 'baobab': ('Baobab', 14.705, -17.460),
    'amitie': ('Amitié', 14.702, -17.455), 'dieuppeul': ('Dieuppeul', 14.718, -17.448),
    'derkle': ('Derklé', 14.722, -17.443), 'castors': ('Castors', 14.712, -17.440),
    'grand dakar': ('Grand Dakar', 14.700, -17.445), 'niary tally': ('Niary Tally', 14.698, -17.440),
    'scat urbam': ('Scat Urbam', 14.732, -17.448), 'zone de captage': ('Zone de Captage', 14.725, -17.438),
    'khar yallah': ('Khar Yallah', 14.728, -17.448), 'grand yoff': ('Grand Yoff', 14.730, -17.445),
    'sicap foire': ('Sicap Foire', 14.735, -17.460), 'nord foire': ('Nord Foire', 14.745, -17.455), 
    'ouest foire': ('Ouest Foire', 14.742, -17.475), 'yoff': ('Yoff', 14.758, -17.463),
    'ouakam': ('Ouakam', 14.723, -17.487), 'mamelles': ('Mamelles', 14.730, -17.500),
    'almadies': ('Almadies', 14.745, -17.518), 'ngor': ('Ngor', 14.752, -17.515),
    'maristes': ('Maristes', 14.725, -17.432), 'parcelles': ('Parcelles', 14.760, -17.430),
    'keur massar': ('Keur Massar', 14.770, -17.310), 'mbao': ('Mbao', 14.745, -17.340)
}

QUARTIERS_TRIES = sorted(COORDS_QUARTIERS_EXTENDED.keys(), key=len, reverse=True)

def distance_haversine(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1): return np.nan
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return round(2 * R * asin(sqrt(a)), 2)

# =====================================================================
# 4. TRAITEMENT DES VALEURS ABERRANTES ET QUARTIERS
# =====================================================================

# Correction Outliers
if 'nb_pieces' in df.columns:
    df.loc[df['nb_pieces'] > 15, 'nb_pieces'] = 4

def corriger_quartier_et_distances(row):
    q_actuel = str(row.get('quartier', 'Inconnu'))
    lat, lon = row.get('latitude', np.nan), row.get('longitude', np.nan)
    
    if q_actuel == 'Inconnu' or pd.isna(lat):
        texte_recherche = str(row.get('url', '')) + " " + str(row.get('titre', '')) + " " + str(row.get('description_complete', ''))
        texte_norm = enlever_accents(texte_recherche).replace("-", " ").replace("'", " ")
        texte_norm = re.sub(r'\b0(\d)\b', r'\1', texte_norm)
        
        for q_nom in QUARTIERS_TRIES:
            if q_nom in texte_norm:
                q_actuel, lat, lon = COORDS_QUARTIERS_EXTENDED[q_nom]
                break
                
    dist_p = distance_haversine(lat, lon, POINTS_INTERET['plateau'][0], POINTS_INTERET['plateau'][1])
    dist_v = distance_haversine(lat, lon, POINTS_INTERET['vdn'][0], POINTS_INTERET['vdn'][1])
    dist_m = distance_haversine(lat, lon, POINTS_INTERET['mer'][0], POINTS_INTERET['mer'][1])
    
    return pd.Series([q_actuel, lat, lon, dist_p, dist_v, dist_m])

df[['quartier', 'latitude', 'longitude', 'dist_plateau_km', 'dist_vdn_km', 'dist_mer_km']] = df.apply(corriger_quartier_et_distances, axis=1)

# Conversions numériques
for col in ['latitude', 'longitude', 'dist_plateau_km', 'dist_vdn_km', 'dist_mer_km']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Exclusion stricte des non-géolocalisés
df = df[(df['quartier'] != 'Inconnu') & (df['latitude'].notna())].copy()

# =====================================================================
# 5. IMPUTATION CONDITIONNELLE DES PIÈCES & DISTANCES
# =====================================================================

# Imputation heuristique du nombre de pièces selon le prix
df['nb_pieces'] = df['nb_pieces'].fillna(
    df['prix_fcfa'].apply(lambda p: 1 if p < 200000 else (2 if p < 350000 else (3 if p < 600000 else 4)))
)

# Imputation des salles de bain
df['nb_salles_bain'] = df['nb_salles_bain'].fillna(
    df['nb_pieces'].apply(lambda p: max(1, int(p) - 1))
)
df['ratio_bains_chambres'] = (df['nb_salles_bain'] / df['nb_pieces']).round(2)

# Imputation géospatiale par la médiane du quartier (avec fallback global)
cols_geo = ['latitude', 'longitude', 'dist_plateau_km', 'dist_vdn_km', 'dist_mer_km']
for col in cols_geo:
    df[col] = df.groupby('quartier')[col].transform(lambda group: group.fillna(group.median()))
    df[col] = df[col].fillna(df[col].median()) # Sécurité globale

# =====================================================================
# 6. SAUVEGARDE ET INSPECTION FINALE (GOLD DATASET)
# =====================================================================

os.makedirs("data/processed", exist_ok=True)
fichier_gold = "data/processed/dataset_dakar_gold.csv"
df.to_csv(fichier_gold, index=False, sep=";")

features_ml = [
    'prix_fcfa', 'nb_pieces', 'nb_salles_bain', 'ratio_bains_chambres',
    'latitude', 'longitude', 'dist_plateau_km', 'dist_vdn_km', 'dist_mer_km',
    'has_groupe', 'has_reserve_eau', 'is_studio', 'is_villa', 'has_piscine',
    'has_ascenseur', 'has_parking', 'has_securite', 'has_clim', 'has_vue_mer',
    'is_meuble', 'has_cuisine_equipee', 'has_balcon', 'is_agence_pro', 'longueur_description'
]

features_presentes = [f for f in features_ml if f in df.columns]

print(f"\n✨ NETTOYAGE GOLD TERMINÉ ! {len(df)} annonces retenues.")
print(f"📊 Nombre de NaN sur les variables du modèle Machine Learning : {df[features_presentes].isna().sum().sum()}")
print(f"📁 Fichier final sauvegardé dans : {fichier_gold}")

🧹 Début du nettoyage final (Zone Gold)...
📦 Annonces ingérées depuis la Zone Silver (data/processed/dataset_dakar_immo.csv) : 3838

✨ NETTOYAGE GOLD TERMINÉ ! 3515 annonces retenues.
📊 Nombre de NaN sur les variables du modèle Machine Learning : 0
📁 Fichier final sauvegardé dans : data/processed/dataset_dakar_gold.csv
